In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import train_test_split, StratifiedKFold
from scipy.optimize import minimize
from tqdm.notebook import tqdm
import matplotlib.font_manager as fm
import numpy as np
import random
%matplotlib inline

# Step 1: Allometric

In [ ]:
# Global random seed (random operations: numpy, pandas, StratifiedKfold, train_test_split)
SEED = 100 # default: 100
random.seed(SEED)
np.random.seed(SEED)

# Configuration
try_num = "HM변형-try1.4-일부"

func = func4
loss_func = loss_func4

lam = 0
test_ratio = 0.3 # default: 0.15
n_fold = 5
params_num = 4
cols = ['DBH(inch)', 'H(ft)', 'CR', 'Cycle']

result_dir = r'D:/ForestFire/CBH/result/Baseline3'
os.makedirs(result_dir, exist_ok=True)


target_trees = [16] # np.unique(df_all.SID) # [code_name_dict[i] for i in valid_species]
rec = np.zeros((len(target_trees), 17 + params_num), dtype=object)
rec[:, 0] = target_trees

# Collector for unified test set
total_test_list = []
total_train_list = []

for i, sid in enumerate(tqdm(target_trees), 1):
    print(f"[{i}/{len(target_trees)}] Processing SID: {sid}")
    condition = (df_all['SID'] == sid)
    nfi6 = df_all.query("Cycle == 6").loc[condition]
    nfi7 = df_all.query("Cycle == 7").loc[condition]

    cnt = (len(nfi6), len(nfi7))
    print(cnt)
    if nfi6.isnull().values.any() or nfi7.isnull().values.any():
        print("Null data detected, skipping SID:", sid)
        continue

    # Split train/test
    if len(nfi6) >= 180:
        nfi6_test = nfi6.sample(frac=test_ratio, random_state=SEED)
        nfi7_test = nfi7.sample(frac=test_ratio, random_state=SEED)
        df_train = pd.concat([nfi6.drop(nfi6_test.index).assign(Cycle=6),nfi7.drop(nfi7_test.index).assign(Cycle=7)])

    elif (len(nfi6) < 180) & (len(nfi7) >= 30):
        # nfi6_test = nfi6
        nfi7_test = nfi7.sample(frac=test_ratio, random_state=SEED)
        df_train = nfi7.drop(nfi7_test.index).assign(Cycle=7)

    else:
        print(f"Skip {sid}: Not enough number of the samples")
        best_params = [np.nan] * params_num
        rec[rec[:, 0] == sid, :] = [
        sid, cnt, np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan,
        np.nan, np.nan, np.nan
        ] + list(best_params)
        continue

    # X, y array 만들기
    X = df_train[cols].drop(columns=['CR'])
    y = df_train['CR']
    stratify_col = df_train['Cycle']

    # score 변수 initialization
    best_score, worst_score, best_params = -np.inf, np.inf, None
    cv_scores = []

    print("lenght of training dataset: ", len(X))
    if len(X) >= 30:
        kf = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=SEED)
        for train_idx, val_idx in kf.split(X, stratify_col):
            X_train = X.iloc[train_idx].drop(columns='Cycle').values.T
            y_train = y.iloc[train_idx].values
            popt, _ = curve_fit(func, X_train, y_train, maxfev=10000)
            result_reg = minimize(loss_func, x0=popt, args=(lam, X_train, y_train))
            opt_params = result_reg.x
            score = r2_score(y_train, func(X_train, *opt_params))
            cv_scores.append(score)
            if score > best_score:
                best_score = score
                best_params = opt_params
            if score < worst_score:
                worst_score = score
    else: continue

    # Evaluate on full training data
    X_full = X.drop(columns='Cycle').values.T
    y_full = y
    y_pred_full = func(X_full, *best_params)
    r2_train = r2_score(y_full, y_pred_full)
    mae_train = mean_absolute_error(y_full, y_pred_full)
    rmse_train = root_mean_squared_error(y_full, y_pred_full)

    # Evaluate on test sets
    # X6_test = nfi6_test[cols].drop(columns=['CR', 'Cycle']).values.T
    # y6_test = nfi6_test['CR']
    X7_test = nfi7_test[cols].drop(columns=['CR', 'Cycle']).values.T
    y7_test = nfi7_test['CR']
        
    # test dataset 예측 (NFI6, NFI7, NFI6+7)
    # y_pred6 = func(X6_test, *best_params)
    y_pred7 = func(X7_test, *best_params)
    
    # test dataset score 산출
    r2_test_nfi6 = np.nan # r2_score(y6_test, y_pred6)
    r2_test_nfi7 = r2_score(y7_test, y_pred7)
    mae_test_nfi6 = np.nan # mean_absolute_error(y6_test, y_pred6)
    mae_test_nfi7 = mean_absolute_error(y7_test, y_pred7)
    rmse_test_nfi6 = np.nan # root_mean_squared_error(y6_test, y_pred6)
    rmse_test_nfi7 = root_mean_squared_error(y7_test, y_pred7)
    y_all_true = y7_test
    y_all_pred = y_pred7
    r2_test_all = r2_score(y_all_true, y_all_pred)
    mae_test_all = mean_absolute_error(y_all_true, y_all_pred)
    rmse_test_all = root_mean_squared_error(y_all_true, y_all_pred)
   
    # test-train dataset list에 저장
    df_train['CR_pred'] = np.clip(y_pred_full, 0, 1)
    total_train_list.extend([df_train])
    nfi6_test['CR_pred'] = np.clip(y_pred6, 0, 1)
    nfi7_test['CR_pred'] = np.clip(y_pred7, 0, 1)
    total_test_list.extend([nfi6_test, nfi7_test])

    # record list에 score 결과 저장
    rec[rec[:, 0] == sid, :] = [
        sid, cnt, np.mean(cv_scores), best_score, worst_score,
        r2_train, mae_train, rmse_train,
        r2_test_all, mae_test_all, rmse_test_all,
        r2_test_nfi6, mae_test_nfi6, rmse_test_nfi6,
        r2_test_nfi7, mae_test_nfi7, rmse_test_nfi7
    ] + list(best_params)

# Save results into a textfile
header = ['SID', 'Count', 'CV_Mean', 'CV_Best', 'CV_Worst',
        'R2_Train', 'MAE_Train', 'RMSE_Train',
        'R2_Test_All', 'MAE_Test_All', 'RMSE_Test_All',
        'R2_Test_NFI6', 'MAE_Test_NFI6', 'RMSE_Test_NFI6',
        'R2_Test_NFI7', 'MAE_Test_NFI7', 'RMSE_Test_NFI7']
np.savetxt(
    os.path.join(result_dir, f'Evaluation_{try_num}.txt'),
    rec, delimiter=',', fmt='%s',
    header=','.join(header + [f'Coef{i+1}' for i in range(11)]),
    comments=''
)

# Save unified test dataset
if total_test_list:
    pd.concat(total_test_list).to_csv(os.path.join(result_dir, f"NFI6+7_test_combined_{try_num}.csv"), index=False, encoding='cp949')
else:
    print("Warning: No test data collected, skipping test dataset save.")

# Save unified train dataset
if total_train_list:
    pd.concat(total_train_list).to_csv(os.path.join(result_dir, f"NFI6+7_train_combined_{try_num}.csv"), index=False, encoding='cp949')
else:
    print("Warning: No test data collected, skipping test dataset save.")

In [ ]:
df_result = pd.DataFrame(rec, columns=header + [f'Par{i}' for i in range(params_num)])
s_names = [reversed_dict[i] for i in df_result['SID']]
df_result.insert(1, 'SName', s_names)
r2_columns = ['CV_Mean', 'CV_Best', 'CV_Worst', 'R2_Train', 'R2_Test_All', 'R2_Test_NFI6', 'R2_Test_NFI7']
df_result.loc[:, r2_columns] = df_result.loc[:, r2_columns].clip(lower=0) # .applymap(lambda x: 0 if x < 0 else x)
df_result.to_csv(os.path.join(result_dir, f'Evaluation_{try_num}.csv'), encoding='cp949')

In [ ]:
df_result.filter(regex="R2_", axis=1)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np

# Set font for Korean
plt.rc('font', family='Malgun Gothic')  
plt.rcParams['axes.unicode_minus'] = False  

# Read data
title = f"Evaluation_{try_num}"
df_result = pd.read_csv(os.path.join(result_dir, title + '.csv'), encoding='cp949')

# Set figure
plt.figure(figsize=(20, 6))
bar_width = 0.15
x = np.arange(len(df_result))

# Plot bars side-by-side
plt.bar(x - 2*bar_width, df_result["CV_Mean"], width=bar_width, label="CV_Mean", color="blue", alpha=0.7)
plt.bar(x - bar_width, df_result["R2_Train"], width=bar_width, label="R2_Train", color="orange", alpha=0.7)
plt.bar(x, df_result["R2_Test_All"], width=bar_width, label="R2_Test_All", color="purple", alpha=0.7)
plt.bar(x + bar_width, df_result["R2_Test_NFI6"], width=bar_width, label="R2_Test_NFI6", color="pink", alpha=0.7)
plt.bar(x + 2*bar_width, df_result["R2_Test_NFI7"], width=bar_width, label="R2_Test_NFI7", color="skyblue", alpha=0.7)

# Annotate values above bars
for i in range(len(df_result)):
    plt.text(x[i] - 2*bar_width, df_result["CV_Mean"][i] + 0.01, f'{df_result["CV_Mean"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] - bar_width, df_result["R2_Train"][i] + 0.01, f'{df_result["R2_Train"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i], df_result["R2_Test_All"][i] + 0.01, f'{df_result["R2_Test_All"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] + bar_width, df_result["R2_Test_NFI6"][i] + 0.01, f'{df_result["R2_Test_NFI6"][i]:.2f}', ha='center', fontsize=6)
    plt.text(x[i] + 2*bar_width, df_result["R2_Test_NFI7"][i] + 0.01, f'{df_result["R2_Test_NFI7"][i]:.2f}', ha='center', fontsize=6)

# X labels and layout
x_labels = [f"{df_result.loc[i, 'SName']}{df_result.loc[i, 'Count']}" for i in range(len(df_result))]
plt.xticks(ticks=x, labels=x_labels, rotation=45, ha="right")
plt.xlabel('SName', fontsize=12)
plt.ylabel("R2 Value", fontsize=12)
plt.title(title, fontsize=14)
plt.legend()
plt.tight_layout()

# Save and show
fig_dir = r'D:/ForestFire/CBH/fig'
plt.savefig(os.path.join(fig_dir, f'{title}.png'))
plt.show()


In [ ]:
# scatter plot
train_header = "NFI6+7_train_combined"
test_header = "NFI6+7_test_combined"
df_train_pred = pd.read_csv(os.path.join(result_dir, f"{train_header}_{try_num}.csv"), encoding="cp949")
df_test_pred = pd.read_csv(os.path.join(result_dir, f"{test_header}_{try_num}.csv"), encoding="cp949")
df_train_pred['Use'] = "train"
df_test_pred['Use'] = "test"
df_merge_pred = pd.concat([df_train_pred, df_test_pred], axis=0)
df_merge_pred.info()

# sns.lmplot(data=df_merge_pred, x="CR", y = "CR_pred", hue="Use", line_kws={"linewidth" : 3, "linestyle" : "--", 'color': 'red'}) #, palette={"pred" : })
for use, df_group in df_merge_pred.groupby("Use"):
    sns.regplot(
        data=df_group,
        x="CR",
        y="CR_pred",
        scatter=True,
        label=f"{use}",
        scatter_kws={'color': 'pink' if use == 'train' else 'lightblue', 's': 30,
                    "alpha" : 0.6 if use == 'train' else 1},
        line_kws={'color': 'blue' if use == 'train' else 'red', 'linewidth': 2, 
                  "alpha" : 0.6 if use == 'train' else 1,
                 "linestyle" : ":" if use == 'train' else "-"}
    )

plt.legend()
plt.show()

# Modeling Building: Tree-based model

In [ ]:
# import ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor

In [ ]:
result_dir = r'D:/ForestFire/CBH/result/Baseline3'
data_file = r"NFI6+7_train_combined_HM변형-try1.0.csv"
df = pd.read_csv(os.path.join(result_dir, data_file), encoding='cp949')
df.info()

In [ ]:
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

In [ ]:
SEED = 200 # 100
random.seed(SEED)
np.random.seed(SEED)

# encoded species code
le = LabelEncoder()
df['SID_ENC'] = le.fit_transform(df['SID'])

feature_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'SID_ENC', 'Lat', 'Long']
cat_col = 'SID'  # species id
target_col = 'CR'  # assuming target column name is 'CR'

# Prepare X, y, species
X = df[feature_cols]
y = df['CR'] - df['CR_pred'] # residuals
species = df[cat_col].astype(str)

# Train/val split stratified by species
X_train, X_val, y_train, y_val, species_train, species_val = train_test_split(
    X, y, species, test_size=0.2, random_state=SEED, stratify=species
)

# Scale only numerical features (except encoded SID)
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
numeric_cols = ['H(ft)', 'DBH(inch)', 'CD(%)', 'Elev(hm)', 'Slope(tan)', 'Azimuth(rad)', 'Lat', 'Long']

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])

# Model definitions
xgb_model = xgb.XGBRegressor(random_state=SEED)

# Fit models
print("Model Training...")
xgb_model.fit(X_train_scaled, y_train)

# Predictions
print("Model prediction...")
residual_pred_train = xgb_model.predict(X_train_scaled)
residual_pred_val = xgb_model.predict(X_val_scaled)
y_final_train = df.loc[X_train.index ,'CR_pred'] + residual_pred_train
y_final_val = df.loc[X_val.index ,'CR_pred'] + residual_pred_val

# Evaluation Function
def evaluate(y_true, y_pred):
    return {
        "R²": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))
    }

# Overall Performance
print("Model Evaluation...")
xgb_metrics = evaluate_model(y_val, xgb_pred)

# Print Overall Comparison
print("Overall Performance Comparison")
eval_result = {
    'Base Model': evaluate(df.loc[X_val.index ,'CR'], df.loc[X_val.index ,'CR_pred']),
    'Residual Model': evaluate(y_val, xgb_res_pred),
    'Hybrid (Final)': evaluate(df.loc[X_val.index ,'CR'], y_final_val)
}
df_eval_result = pd.DataFrame(eval_result).T

# Per-Species Evaluation
unique_species = species_val.unique()
records = []
df_val = df.loc[X_val.index]
for sid in unique_species:
    idx = (species_val == sid)
    if idx.sum() > 1:  # skip species with <2 samples
        record = {"SID": sid}
        # XGBoost
        eval_result = {
        'Base Model': evaluate(df_val.loc[idx, 'CR'], df_val.loc[idx, 'CR_pred']),
        'Residual Model': evaluate(y_val[idx], xgb_res_pred[idx]),
        'Hybrid (Final)': evaluate(df_val.loc[idx, 'CR'], y_final_val[idx])
         }
        df_eval_result = pd.DataFrame(eval_result).T
        df_eval_result['SID'] = sid
        df_eval_result = df_eval_result.set_index('SID', append=True)
        records.append(df_eval_result)

df_species_comparison = pd.concat(records, axis=0)
df_species_comparison.index.names = ['Model', 'SID']
df_species_comparison = df_species_comparison.reorder_levels(['SID', 'Model'])
df_species_comparison

In [ ]:
df_species_comparison.to_csv(os.path.join(result_dir, "Result_eval_residual_1.0.csv"))

In [ ]:
# feature importance
import matplotlib.pyplot as plt
import numpy as np

# Feature names
feature_names = X_train_scaled.columns.tolist()

# Get feature importances
xgb_importance = xgb_model.feature_importances_

# Sort features by XGBoost importance (for consistent display)
sorted_idx = np.argsort(xgb_importance)[::-1]
sorted_features = [feature_names[i] for i in sorted_idx]

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

bar_width = 0.35
x = np.arange(len(feature_names))

bars1 = ax.bar(x - bar_width/2, xgb_importance[sorted_idx], width=bar_width, label='XGBoost')

# Add value labels
for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax.annotate(f"{height:.2f}", xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)

# Final touches
ax.set_xticks(x)
ax.set_xticklabels(sorted_features, rotation=45, ha='right')
ax.set_ylabel("Feature Importance")
ax.set_title("Feature Importance")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_species_comparison_test = pd.DataFrame(records_test).sort_values(by="SID").reset_index(drop=True)
# df_species_comparison.columns = pd.MultiIndex.from_product([['Train'], df_species_comparison.columns])
df_species_comparison_test.columns = pd.MultiIndex.from_product([['Test'], df_species_comparison_test.columns])
df_comparison_merged = pd.concat([df_species_comparison, df_species_comparison_test], axis=1)
df_comparison_merged

# Hyperparameter tuning with XGBoost